# Team Standings Model

In [1]:
# For data handling
import pandas as pd
import numpy as np

# To manage model loading and save paths
import os
import json
import joblib
from datetime import datetime

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model building
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler


# Machine Learning
from sklearn import tree, svm
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from pytorch_tabnet.tab_model import TabNetClassifier

In [5]:
# Creating a timestamp for saving model version number by date
timestamp = datetime.now().strftime("%Y_%m_%d")

# Establishing file paths for loading and saving data
CLEANED_PATH = "../../data/teams_cleaned"
PROCESSED_PATH = "../../data/processed"
ML_PATH = "../../data/models/ml"
DL_PATH = "../../data/models/dl"
os.makedirs(ML_PATH, exist_ok=True)
os.makedirs(DL_PATH, exist_ok=True)

# Create a timestamped subdirectory inside DL and ML Path
versioned_dl_path = os.path.join(DL_PATH, f"best_team_{timestamp}")
os.makedirs(versioned_dl_path, exist_ok=True)

versioned_ml_path = os.path.join(ML_PATH, f"best_team_{timestamp}")
os.makedirs(versioned_ml_path, exist_ok=True)

In [3]:
team_df = pd.read_csv(f"{PROCESSED_PATH}/team_eda_cleaned.csv")

In [4]:
team_df.drop(columns=["Unnamed: 0"], inplace=True)

In [6]:
all_seasons_df = pd.read_csv(f"{CLEANED_PATH}/team_all_years.csv")

In [7]:
team_df.index.equals(all_seasons_df.index)

True

In [11]:
# Creating a helper column for determing the best record in a season
all_seasons_df["Max_Wins"] = all_seasons_df.groupby("Season")["W"].transform("max")

In [12]:
# Creating the has_Best_Record column
all_seasons_df["has_Best_Record"] = all_seasons_df["W"] == all_seasons_df["Max_Wins"]

In [13]:
# Dropping the Max Wins column as it's no longer needed
all_seasons_df.drop(columns=["Max_Wins"], inplace=True)

In [14]:
selected_features = list(team_df.drop(columns=["has_Best_Record"]).columns)
reduced_with_season = all_seasons_df[selected_features + ["Season", "has_Best_Record"]].dropna()

In [15]:
# Checking to see if indexes match before merging
team_df.index.equals(reduced_with_season.index)

True

In [16]:
# Merging the DataFrames
team_df["Season"] = reduced_with_season["Season"]